# Multi-Headed Attention in a Transformer

**Full multi-head attention module at the bottom.**

---

Instead of just running a single attention block, we can split the input dimension into heads. \
This allows enables multiple heads to work in parallel, each learning different representations and patterns.

This process comes with only a marginal increase in computational cost, as $d_{model}$ is **split across each head**, instead of each head containing all $d_{model}$ feature.

Data pipeline:

input tensor $X:[B,T,d_{model}]$ \
↓ \
projection matrices $W_Q,W_K,W_V$ (and $W_O$) \
↓ \
compute $Q,K,V$ \
↓ \
split into heads $Q,K,V:[B,H,T,d_{head}]$ \
↓ \
perform RoPE to $Q,K$, with $V$ unchanged \
↓ \
attention scores $Q \cdot K^T$ \
↓ \
scale by $\sqrt {d_{head}}$ \
↓ \
causal mask \
↓ \
normalize with softmax \
↓ \
weighted sums $V$ \
↓ \
tranpose $[B,T,H,d_{head}]$ \
↓ \
concatenate $[B,T,d_{model}]$ \
↓ \
ouput projection $W_O$

---

Single-head: \
$[B, T, d_{model}]$

**Multi**-head: \
$[B, T, d_{model}] \rightarrow \ $ split $d_{model}$ across heads $\ \rightarrow [B, H, T, d_{head}]$, where $d_{head} = \frac{d_{model}}{H}$

In [244]:
import torch
import torch.nn as nn
import math

## Initialize $Q,K,V$

To work with an inspectable input, a relatively low $H$ size will be used. 

In [245]:
batch_size = 1 # B
sequence_length = 4 # T
d_model = 16 # d_model
num_heads = 2 # H

head_dim = d_model // num_heads # d_head

print("head_dim:", head_dim)

head_dim: 8


In [246]:
# Random input data
x = torch.randn(
    batch_size,
    sequence_length,
    d_model
)

print("x shape:", x.shape)

x shape: torch.Size([1, 4, 16])


$Q, V, K$ projections. Typically $W_O$ is initialized here as well.

In [247]:
q_proj = nn.Linear(d_model, d_model, bias=False)
k_proj = nn.Linear(d_model, d_model, bias=False)
v_proj = nn.Linear(d_model, d_model, bias=False)

q = q_proj(x)
k = k_proj(x)
v = v_proj(x)

print("Q shape:", q.shape)
print("K shape:", k.shape)
print("V shape:", v.shape)

Q shape: torch.Size([1, 4, 16])
K shape: torch.Size([1, 4, 16])
V shape: torch.Size([1, 4, 16])


## Important Model Decision

### Attention Projection Biases

The $Q, K, V$, and output $W_O$ projections may include learned bias terms.

Using biases adds a small number of parameters and slightly more flexibility,
while **omitting** them simplifies the projections and is common in some modern
decoder-only architectures.

**Whether biases should be applied will be a decision made during the final stages of coding the LLM.**

## Split $Q,V,K$

Before this, nothing is multi-head yet.

---

We now split the last dimension of the tensors $[B,T,\mathbf{d_{model}}]$:

$d_{model}$

↓

$H$ and $d_{head}$

---

Pytorch's `view()` function changes the shape of a tensor **without changing its underlying data**. Batch size $B$ and sequence length $T$ are kept untouched.

https://docs.pytorch.org/docs/2.14/generated/torch.Tensor.view.html

In [248]:
# Split last dimension into multiple heads
q = q.view(
    batch_size, # B
    sequence_length, # T
    num_heads, # H
    head_dim # d_head
)

print("Q after view:", q.shape)

Q after view: torch.Size([1, 4, 2, 8])


Currently $Q$ has shape $[B,T,H,d_{head}]$. Multi-head attention math requires each of the $H$ heads has its own sequence of $T$ tokens, each represented by $d_{head}$ features.

---

Tranposing the **second** and **third** dimensions $[B,\mathbf{T,H},d_{head}] \rightarrow [B,\mathbf{H,T},d_{head}]$ does exactly that. Each **head** now contains a sequence of $d_{head}$-dimensional token (vector) representations.

In [249]:
# Tranpose second and third dimensions
q = q.transpose(1, 2)

print("Q after transpose:", q.shape)

Q after transpose: torch.Size([1, 2, 4, 8])


So now instead of computing one attention head calculation over $d_{model}$ features, multiple heads can now be computed. \
PyTorch computes each head in **parallel** because the head dimesion is a part of the tensor.

---

$K$ and $V$ are also transformed.

In [250]:
# Perform same split and tranpose operations on K and V
k = k.view(
    batch_size,
    sequence_length,
    num_heads,
    head_dim
).transpose(1, 2)

v = v.view(
    batch_size,
    sequence_length,
    num_heads,
    head_dim
).transpose(1, 2)

print("Q:", q.shape)
print("K:", k.shape)
print("V:", v.shape)

Q: torch.Size([1, 2, 4, 8])
K: torch.Size([1, 2, 4, 8])
V: torch.Size([1, 2, 4, 8])


## Important Model Decision

### Number of Attention Heads

The number of attention heads $H$ determines how many **separate attention patterns** the model can learn in parallel. The model dimension is split **evenly** across the heads $d_{head} = \frac{d_{model}}{H}$.

$H$ **must** divide $d_{model}$ evenly, for example:

$d_{model}=512,\  H = 8\  \rightarrow \ d_{head} = \frac{512}{8} = 64$

---

More heads means more separate attention patterns, but also makes each head **smaller**. Too many heads can leave each head with **too few features** to represent useful information.

A practical target is keeping $d_{head}$ around 32–64, with 64 being a very common choice. \
So to meet that $d_{head}$ size target:

$d_{model}=512$ would have $H = 8$ \
$d_{model}=384$ would have $H = 6$ \
$d_{model}=256$ would have $H = 4$

**The final number of heads should therefore be chosen together with the model dimension rather than independently.**

---

### Important Computational Nuance

As mentioned before, multiple heads means splitting the same $d_{model}$-wide $Q,K,V$ representations (evenly) across smaller heads, not creating $H$ full $d_{model}$-sized attention blocks.

Increasing $H$ does **not** substantially increase the parameter count of standard multi-head attention as long as $d_{model}$ stays fixed. \
This is one of the main reasons why multi-head attention is so effective and vital.

## Splitting And Computing Attention Scores For Multiple Heads

The same process of computing the scores for a single head is perform, but now runs independently acrosse every head because of the extra head $H$ dimension.

---

#### Tensor Shapes

Input $X: [1,4,16]$ \
$Q, K, V: [1, 2, 8, 4]$

In [251]:
# Dot product of Q and K^T
scores = q @ k.transpose(-2, -1)

print("Q shape:", q.shape)
print("K^T shape:", k.transpose(-2, -1).shape)

print("\nScores shape:", scores.shape)

Q shape: torch.Size([1, 2, 4, 8])
K^T shape: torch.Size([1, 2, 8, 4])

Scores shape: torch.Size([1, 2, 4, 4])


For each head they get a $[T,T]$ attention matrix.

---

The raw scores for each head are scaled by $\sqrt{d_{head}}$ instead of $\sqrt{d_{k}}$, as each head is only comparing ${d_{head}}$-dimensional vectors.

In [252]:
scaled_scores = scores / math.sqrt(head_dim)

print("Scaled scores shape:", scaled_scores.shape)

Scaled scores shape: torch.Size([1, 2, 4, 4])


Causal mask is implemented and applied before softmax, PyTorch **broadcasts** the mask automatically.

In [253]:
mask = torch.tril(
    torch.ones(
        sequence_length,
        sequence_length,
        dtype=torch.bool # boolean mask
    )
)

masked_scores = scaled_scores.masked_fill(
    ~mask, # inverse booleans
    float("-inf")
)

Now softmax is applied. Verifying now shows $2$ heads, with $4$ rows that have features that sum to $1$.

In [254]:
attention_weights = torch.softmax(
    masked_scores,
    dim=-1 # softmax across last dim
)

print("Attention weights shape:", attention_weights.shape)

print("\nCheck every row in each head sum to 1:\n", attention_weights.sum(dim=-1))

Attention weights shape: torch.Size([1, 2, 4, 4])

Check every row in each head sum to 1:
 tensor([[[1., 1., 1., 1.],
         [1., 1., 1., 1.]]], grad_fn=<SumBackward1>)


The weighted sums are calculated, outputting a tensor with shape $[B,H,T,d_{head}]$.

In [255]:
head_outputs = attention_weights @ v

print("Head outputs shape:", head_outputs.shape)

Head outputs shape: torch.Size([1, 2, 4, 8])


## Merging The Heads Back

Each head outputs a $[B,H,T,d_{head}]$ shaped tensor, but the transformer block expects the final output to be shaped $[B,T,d_{model}]$.

The **concatenation** process is essentially just reversed.

---

Since $d_{head} = \frac{d_{model}}{H}$, the reverse is true:

$d_{model} = H \times d_{head}$

In [256]:
# Tranpose second and third dimensions
combined = head_outputs.transpose(1, 2)

print("After transpose:", combined.shape)

After transpose: torch.Size([1, 4, 2, 8])


Now the shape is $[B, T, H, d_{head}]$. The $H$ and $d_{head}$ dimensions are merged back.

---

We use `reshape()` to recombine the heads because the preceding `transpose()` operations may leave the tensor **non-contiguous** in memory (memory layout of tensor changed `view()` won't work).

https://docs.pytorch.org/docs/2.14/generated/torch.reshape.html

In [257]:
combined = combined.reshape(
    batch_size, # B
    sequence_length, # T
    d_model # H * d_head = d_model
)

print("Combined shape:", combined.shape)

Combined shape: torch.Size([1, 4, 16])


## Final $W_O$ Projection

A final **learned output projection** matrix $W_O$ is used to transform the concatenated head output.

$W_O$ is typically initialized with $W_Q, W_K, W_V$ in production code.

---

After concatenation, the heads are technically one single $d_{model}$-sized vector again, but their information is still essentially placed "side-by-side". \
Conceptually, $W_O$ mixes the information across the heads into a **unified representation**.

In [258]:
# W_O
out_proj = nn.Linear(
    d_model,
    d_model,
    bias=False
)

# W_O * combined
output = out_proj(combined)

# Input/output shape sanity check
assert output.shape == x.shape

print("Final output shape:", output.shape)

Final output shape: torch.Size([1, 4, 16])


This wraps up the entire causal multi-head attention block, which can be mathmematically denoted:

#### $\text{MultiHead}(Q,K,V) = \text{Concat}(head_1,\ldots,head_H)W_O$

In [259]:
class MultiHeadCausalAttention(nn.Module):
    """
    Multi-head causal self-attention
    """

    def __init__(self, d_model, num_heads):
        super().__init__()

        # model dim must be divisible by number of heads
        assert d_model % num_heads == 0

        self.d_model = d_model # d_model
        self.num_heads = num_heads # H
        self.head_dim = d_model // num_heads # d_head = d_model / H

        self.q_proj = nn.Linear(d_model, d_model, bias=False)
        self.k_proj = nn.Linear(d_model, d_model, bias=False)
        self.v_proj = nn.Linear(d_model, d_model, bias=False)

        self.out_proj = nn.Linear(d_model, d_model, bias=False) # W_O

    def forward(self, x):
        batch_size, sequence_length, _ = x.shape

        # Project input into Q, K, V
        q = self.q_proj(x)
        k = self.k_proj(x)
        v = self.v_proj(x)

        # Split d_model across attention heads
        q = q.view(
            batch_size,
            sequence_length,
            self.num_heads,
            self.head_dim
        ).transpose(1, 2) # [B, H, T, d_head]

        k = k.view(
            batch_size,
            sequence_length,
            self.num_heads,
            self.head_dim
        ).transpose(1, 2)

        v = v.view(
            batch_size,
            sequence_length,
            self.num_heads,
            self.head_dim
        ).transpose(1, 2)

        # Scaled dot product attention
        scores = q @ k.transpose(-2, -1)
        scores = scores / math.sqrt(self.head_dim)

        # Causal mask
        mask = torch.tril(
            torch.ones(
                sequence_length,
                sequence_length,
                device=x.device,
                dtype=torch.bool
            )
        )
        
        # Apply mask
        scores = scores.masked_fill(
            ~mask,
            float("-inf")
        )

        # Normalize with softmax
        attention_weights = torch.softmax(
            scores,
            dim=-1
        )

        # Weighted sum of V
        head_outputs = attention_weights @ v

        # [B, H, T, D_h] -> [B, T, H, D_h]
        head_outputs = head_outputs.transpose(1, 2)

        # Concatenate heads
        combined = head_outputs.reshape(
            batch_size,
            sequence_length,
            self.d_model
        )

        # Final output projection W_O
        output = self.out_proj(combined)

        return output

Testing the class with the initial model dimensions and number of heads. Output and Input tensor shapes should match.

In [260]:
attention = MultiHeadCausalAttention(
    d_model=d_model,
    num_heads=num_heads
)

output = attention(x)

assert output.shape == x.shape

print("Input shape:", x.shape)
print("Output shape:", output.shape)

Input shape: torch.Size([1, 4, 16])
Output shape: torch.Size([1, 4, 16])


## Rotary Positional Encoding

RoPE is a positional encoding method typically applied to $Q,K$ after they've been split into attention heads. It offers the benefits of absolute PE without a lot of the drawbacks.

$V$ does **not** get encoded because it essentially contains the **information about the vector**, it doesn't need positional encoding for the attention comparison.

---

**Note: $d_{head} must be even in order for RoPE to work**

RoPE groups **feature** dimensions, for example: \
$X \rightarrow [x_0,x_1],[x_2,x_3], ... ,[x_{n-1},x_n]$

And each **pair of dimensions** gets rotated by a **position** dependent angle:

$\begin{bmatrix} x'_{even} \\ x'_{odd} \end{bmatrix} = \begin{bmatrix} \cos\theta & -\sin\theta \\ \sin\theta & \cos\theta \end{bmatrix} \begin{bmatrix} x_{even} \\ x_{odd} \end{bmatrix}$ or equivalently 
$\begin{aligned}
x'_{even} &= x_{even}\cos\theta - x_{odd}\sin\theta \\
x'_{odd} &= x_{even}\sin\theta + x_{odd}\cos\theta
\end{aligned}$

Where $\theta$ is determined by **token position** and which **feature pair is being rotated**.

---

RoPE pairs up dimensions (features) within each $Q,K$ vector and **temporarily treats** each pair as a tiny 2D vector that can be rotated. This can be misleading as the pair of dimensions **aren't separate vectors**, just treated as such for the sake of RoPE.

Below is a basic example of rotating a pair of even/odd dimensions by 90 degrees.

In [261]:
x_even = torch.tensor([1.0])
x_odd = torch.tensor([0.0])

theta = torch.tensor([math.pi / 2]) # 90 degrees (counter-clockwise)

# (x_even * cosθ) - (x_odd * sinθ)
rotated_even = (
    x_even * torch.cos(theta)
    - x_odd * torch.sin(theta)
)

# (x_even * sinθ) + (x_odd * cosθ)
rotated_odd = (
    x_even * torch.sin(theta)
    + x_odd * torch.cos(theta)
)

print("Even dimension:", rotated_even)
print("Odd dimension:", rotated_odd)

Even dimension: tensor([-4.3711e-08])
Odd dimension: tensor([1.])


The even dimension should have a value ~0 (slightly off because of floating point precision) and the odd dimension 1.

Roughly: \
$[1,0] \rightarrow [0,1]$

---

### Actual Code

A common RoPE formula used is: \
$\theta_{p,i} = p \cdot \frac{1}{base^{2i/d}}$

Where \
$p$: token position \
$i$: feature pair index \
$d$: head dimension $d_{head}$ \
base ≈ 10000

That means different feature pairs rotate at different speeds depending on $p, i, d$ values. \
So some dimensions encode faster-changing positional relationships, while others change more slowly.

https://docs.pytorch.org/docs/2.14/generated/torch.arange.html

In [262]:
def build_rope_frequencies(head_dim, base=10000):
    assert head_dim % 2 == 0

    pair_indices = torch.arange(
        0, # start
        head_dim, # end
        2, # step (pairs)
        dtype=torch.float32
    )

    inverse_frequencies = 1.0 / (
        base ** (pair_indices / head_dim) # base^(i/d)
    )

    return inverse_frequencies

Using our initial $d_{head}$ of $8$ we can see how pairs of dimensions now get different rotational frequencies.

High frequency pairs get local positional information, while lower frequency pairs encode longer-range relationships between tokens.

In [263]:
head_dim = 8

inverse_frequencies = build_rope_frequencies(head_dim)

print("Inverse frequencies:", inverse_frequencies)
print("\nShape:", inverse_frequencies.shape)

Inverse frequencies: tensor([1.0000, 0.1000, 0.0100, 0.0010])

Shape: torch.Size([4])


Now the actual implementation of the equation, multiplying $p \cdot \frac{1}{base^{2i/d}}$

In [264]:
def build_rope_angles(sequence_length, head_dim, base=10000):
    # 1 / base^(i/d)
    inverse_frequencies = build_rope_frequencies(
        head_dim,
        base
    )

    # p
    positions = torch.arange(
        sequence_length,
        dtype=torch.float32
    )

    # p * (1 / base^(i/d))
    angles = positions[:, None] * inverse_frequencies[None, :] # None for broadcasting

    return angles

We'll now get the angles in the form of shape:

$\cos,\sin:[T, \frac{d_{head}}{2}]$

---

Each **row** corresponds to a token position $T$, and each **column** corresponds to a feature **pair** being rotated, hence $\frac{d_{head}}{2}$.

In [265]:
angles = build_rope_angles(
    sequence_length=4,
    head_dim=8
)

print(angles)
print("Shape:", angles.shape)

tensor([[0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00],
        [1.0000e+00, 1.0000e-01, 1.0000e-02, 1.0000e-03],
        [2.0000e+00, 2.0000e-01, 2.0000e-02, 2.0000e-03],
        [3.0000e+00, 3.0000e-01, 3.0000e-02, 3.0000e-03]])
Shape: torch.Size([4, 4])


In [266]:
cos = torch.cos(angles)
sin = torch.sin(angles)

print("cos shape:", cos.shape)
print("sin shape:", sin.shape)

cos shape: torch.Size([4, 4])
sin shape: torch.Size([4, 4])


We can wrap the operations for the even and odd indices in a function that applies the rotation transformations.

https://docs.pytorch.org/docs/2.14/generated/torch.empty_like.html

In [267]:
def apply_rope(x, cos, sin):
    # Split last dim into even/odd
    x_even = x[..., 0::2] # ... ellipsis means keep all preceding dimensions unchanged
    x_odd = x[..., 1::2]

    # (x_even * cosθ) - (x_odd * sinθ)
    rotated_even = (
        x_even * cos
        - x_odd * sin
    )

    # (x_even * sinθ) + (x_odd * cosθ)
    rotated_odd = (
        x_even * sin
        + x_odd * cos
    )

    rotated = torch.empty_like(x) # input-shaped tensor with uninitialized values

    # Fill tensor's even/odd indices
    rotated[..., 0::2] = rotated_even
    rotated[..., 1::2] = rotated_odd

    return rotated

Because tensor shape $\cos,\sin:[T, \frac{d_{head}}{2}]$, we need it to resemble $Q,K: [B, H, T, d_{head}]$.

Pytorch's `unsqueeze()` adds another dimension without changing the values inside.

https://docs.pytorch.org/docs/2.14/generated/torch.unsqueeze.html

In [268]:
# Add batch and head dimensions
cos = cos.unsqueeze(0).unsqueeze(0)
sin = sin.unsqueeze(0).unsqueeze(0)

print(cos.shape)

torch.Size([1, 1, 4, 4])


In [269]:
q_rope = apply_rope(q, cos, sin)
k_rope = apply_rope(k, cos, sin)

assert q.shape == q_rope.shape

print("Q before:", q.shape)
print("Q after RoPE:", q_rope.shape)

print("K before:", k.shape)
print("K after RoPE:", k_rope.shape)

Q before: torch.Size([1, 2, 4, 8])
Q after RoPE: torch.Size([1, 2, 4, 8])
K before: torch.Size([1, 2, 4, 8])
K after RoPE: torch.Size([1, 2, 4, 8])


Then the attention process continues after i.e. calculating $QK^T$ and beyond...

RoPE fits in the attention pipeline:

$...$ \
↓ \
split into heads $Q,K,V:[B,H,T,d_{head}]$ \
↓ \
**apply RoPE to $Q,K$ only, $V$ unchanged** \
↓ \
attention scores $Q \cdot K^T$ \
↓ \
$...$

## Putting Everything Together

In [270]:
def build_rope_angles(sequence_length, head_dim, device, base=10000):
    """
    Build angles for RoPE.
    """
    
    assert head_dim % 2 == 0 # must be even

    pair_indices = torch.arange(
        0,
        head_dim,
        2, # pairs
        dtype=torch.float32,
        device=device # ensure tensors created on same device
    )

    # 1 / base^(i/d)
    inverse_frequencies = 1.0 / (
        base ** (pair_indices / head_dim)
    )

    # p
    positions = torch.arange(
        sequence_length,
        dtype=torch.float32,
        device=device
    )

    # p * (1 / base^(i/d))
    angles = positions[:, None] * inverse_frequencies[None, :] # broadcasted

    return torch.cos(angles), torch.sin(angles)

In [271]:
def apply_rope(x, cos, sin):
    """
    Applying RoPE.
    """

    # Split last dim indices into even/odd
    x_even = x[..., 0::2]
    x_odd = x[..., 1::2]

    # Add batch and head dimensions
    cos = cos.unsqueeze(0).unsqueeze(0)
    sin = sin.unsqueeze(0).unsqueeze(0)

    # (x_even * cosθ) - (x_odd * sinθ)
    rotated_even = (
        x_even * cos
        - x_odd * sin
    )

    # (x_even * sinθ) + (x_odd * cosθ)
    rotated_odd = (
        x_even * sin
        + x_odd * cos
    )

    rotated = torch.empty_like(x)

    # Fill indices with correponding even/odd values
    rotated[..., 0::2] = rotated_even
    rotated[..., 1::2] = rotated_odd

    return rotated

Updated `MultiHeadCausalAttention` class.

In [272]:
class MultiHeadCausalAttention(nn.Module):
    """
    Multi-head causal self-attention with RoPE.
    """

    def __init__(self, d_model, num_heads):
        super().__init__()

        assert d_model % num_heads == 0

        self.d_model = d_model
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads

        assert self.head_dim % 2 == 0

        self.q_proj = nn.Linear(d_model, d_model, bias=False)
        self.k_proj = nn.Linear(d_model, d_model, bias=False)
        self.v_proj = nn.Linear(d_model, d_model, bias=False)

        self.out_proj = nn.Linear(d_model, d_model, bias=False)

    def forward(self, x):
        batch_size, sequence_length, _ = x.shape

        # Q, K, V projections
        q = self.q_proj(x)
        k = self.k_proj(x)
        v = self.v_proj(x)

        # Split into attention heads
        q = q.view(
            batch_size,
            sequence_length,
            self.num_heads,
            self.head_dim
        ).transpose(1, 2)

        k = k.view(
            batch_size,
            sequence_length,
            self.num_heads,
            self.head_dim
        ).transpose(1, 2)

        v = v.view(
            batch_size,
            sequence_length,
            self.num_heads,
            self.head_dim
        ).transpose(1, 2)

        # Get RoPE angles
        cos, sin = build_rope_angles(
            sequence_length,
            self.head_dim,
            x.device
        )

        # Apply RoPE to Q and K
        q = apply_rope(q, cos, sin)
        k = apply_rope(k, cos, sin)

        # Dot product attention
        scores = q @ k.transpose(-2, -1)

        # Scale raw scores
        scores = scores / math.sqrt(self.head_dim)

        # Initialize causal mask
        mask = torch.tril(
            torch.ones(
                sequence_length,
                sequence_length,
                dtype=torch.bool,
                device=x.device
            )
        )

        # Apply mask
        scores = scores.masked_fill(
            ~mask,
            float("-inf")
        )

        # Softmax normalization
        attention_weights = torch.softmax(
            scores,
            dim=-1
        )

        # Weighted sum of values
        head_outputs = attention_weights @ v

        # Concatenate heads
        combined = (
            head_outputs
            .transpose(1, 2) # [B, T, H, d_head]
            .reshape( # concatenate heads
                batch_size,
                sequence_length,
                self.d_model
            )
        )

        # Mix information across heads
        output = self.out_proj(combined)

        return output

Testing the class.

In [273]:
attention = MultiHeadCausalAttention(
    d_model=16,
    num_heads=2
)

x = torch.randn(1, 4, 16)

output = attention(x)

assert output.shape == x.shape

print("Input shape:", x.shape)
print("Output shape:", output.shape)

Input shape: torch.Size([1, 4, 16])
Output shape: torch.Size([1, 4, 16])


## Final Verdict

Multi-head attention splits $d_{model}$ across multiple heads, each performing **causal scaled dot-product attention**. RoPE adds **positional information** to $Q,K$ before attention.

The heads are concatenated and mixed through $W_O$, returning tensor $[B, T, d_model]$.

$d_{model}, H$, biases, and related hyperparameters remain in **unfinalized**.